In [2]:
import pandas as pd
import joblib

import matplotlib.pyplot as plt

import os

import shap

import numpy as np

### Load models

In [3]:
# Note: _sm stands for a model trained on SMOTE-enhanced data

# Logistic Regression
lr = joblib.load('models/logistic_regression_best.pkl')

# Decision Tree
dec_tree = joblib.load('models/decision_tree_best.pkl')

# Random Forest
rf = joblib.load('models/best_rf.pkl')

# XGBoost
xgb = joblib.load('models/best_xgb.pkl')

# LightGBM
lgbm_sm = joblib.load('models/best_lightgbm_sm.pkl')

### Load necessary datasets

In [4]:
X_train_lr = pd.read_parquet('X_train_lr.parquet')
y_train_lr = pd.read_parquet('y_train_lr.parquet')
X_test_lr = pd.read_parquet('X_test_lr.parquet')
y_test_lr = pd.read_parquet('y_test_lr.parquet')

X_train_others = pd.read_parquet('X_train_others.parquet')
y_train_others = pd.read_parquet('y_train_others.parquet')
X_train_others_smote = pd.read_parquet('X_train_others_smote.parquet')
y_train_others_smote = pd.read_parquet('y_train_others_smote.parquet')
X_test_others = pd.read_parquet('X_test_others.parquet')
y_test_others = pd.read_parquet('y_test_others.parquet')

### Global understandability
Extract 10 most important features and their influence for each model.

In [52]:
# Create a folder
os.makedirs('model_results/understandability', exist_ok=True)

#### Logistic Regression

In [5]:
coef = pd.Series(lr.coef_[0], index=X_train_lr.columns)

global_importance = (
    coef.abs()
        .sort_values(ascending=False)
        .head(10)
)

coef.loc[global_importance.index]

ext_source_3                  -0.499258
ext_source_2                  -0.378572
ext_source_1                  -0.197557
flag_document_3                0.148768
days_employed                  0.146479
ext_source_1_missing           0.126811
ext_source_3_missing           0.093821
amt_annuity                    0.088513
flag_document_6                0.085189
region_rating_client_w_city    0.080613
dtype: float64

#### Decision Tree

In [6]:
importance = pd.Series(
    dec_tree.feature_importances_,
    index=X_train_others.columns
)

global_importance = (
    importance
        .sort_values(ascending=False)
        .head(10)
)

global_importance

ext_source_3        0.512293
ext_source_2        0.388358
ext_source_1        0.074279
days_employed       0.025070
flag_own_realty     0.000000
amt_income_total    0.000000
amt_annuity         0.000000
cnt_children        0.000000
amt_credit          0.000000
flag_own_car        0.000000
dtype: float64

#### Random Forest

##### Create a sample of X_train_others to make things computationally easier

In [7]:
# Sample the test set for faster SHAP computation
X_sample = X_test_others.sample(n=2000, random_state=42)

In [8]:
explainer_rf = shap.TreeExplainer(rf)

shap_values = explainer_rf(X_sample)


In [9]:
global_importance_rf = pd.Series(
    np.abs(shap_values.values[:, :, 1]).mean(axis=0),
    index=X_sample.columns
).sort_values(ascending=False).head(10)

global_importance_rf

ext_source_3                                         0.050670
ext_source_2                                         0.049884
ext_source_1                                         0.021827
days_employed                                        0.019164
days_birth                                           0.016815
name_education_type_Higher education                 0.013307
days_last_phone_change                               0.012968
amt_credit                                           0.010632
code_gender_M                                        0.010425
name_education_type_Secondary / secondary special    0.009594
dtype: float64

In [ ]:
os.makedirs("model_results", exist_ok=True)
joblib.dump(shap_values, "model_results/understandability/shap_values_rf.pkl")

['model_results/shap_values_rf.pkl']

#### XGBoost

In [11]:
explainer_xgb = shap.TreeExplainer(xgb)

shap_values_xgb = explainer_xgb(X_sample)

global_importance_xgb = pd.Series(
    np.abs(shap_values_xgb.values).mean(axis=0),
    index=X_sample.columns
).sort_values(ascending=False).head(10)

global_importance_xgb

ext_source_3                            0.390196
ext_source_2                            0.328251
ext_source_1                            0.142337
days_employed                           0.122651
amt_credit                              0.110593
days_birth                              0.089298
amt_annuity                             0.083530
name_education_type_Higher education    0.075341
days_id_publish                         0.072400
code_gender_M                           0.065194
dtype: float32

In [ ]:
joblib.dump(shap_values_xgb, "model_results/understandability/shap_values_xgb.pkl")

['model_results/shap_values_xgb.pkl']

#### LightGBM

In [53]:
explainer_lgbm = shap.TreeExplainer(lgbm_sm)

shap_values_lgbm = explainer_lgbm(X_sample)

global_importance_lgbm = pd.Series(
    np.abs(shap_values_lgbm.values).mean(axis=0),
    index=X_sample.columns
).sort_values(ascending=False).head(10)

global_importance_lgbm

ext_source_3                  0.668572
name_income_type_Working      0.367737
ext_source_2                  0.357196
ext_source_3_missing          0.284153
ext_source_1_missing          0.237638
code_gender_M                 0.212829
name_family_status_Married    0.160076
flag_document_3               0.140713
reg_or_liv_city_not_work      0.138271
code_gender_F                 0.130252
dtype: float64

In [54]:
joblib.dump(shap_values_lgbm, "model_results/understandability/shap_values_lgbm.pkl")

['model_results/understandability/shap_values_lgbm.pkl']

### Local understandability

#### Methodology
A true positive (tp), true negative (tn), false positive (fp) and false negative (fn) observation will be randomly sampled for each model. The 5 most influential features for each observation will be extracted (coefficients for LR and DT and with SHAP analysis for the other models).

#### Logistic Regression 

In [14]:
# Create predictions
y_pred = lr.predict(X_test_lr)

comparison = X_test_lr.copy()
comparison["Actual"] = y_test_lr.values
comparison["Prediction"] = y_pred

In [15]:
# Randomly sample tp, tn, fp, fn 
tp = comparison[
    (comparison.Actual == 1) &
    (comparison.Prediction == 1)
].sample(1, random_state=42)
tp = tp[X_test_lr.columns] # Choose this to remove the "Actual" and "Prediction" columns from the tp dataframe

fp = comparison[
    (comparison.Actual == 0) &
    (comparison.Prediction == 1)
].sample(1, random_state=42)
fp = fp[X_test_lr.columns]

tn = comparison[
    (comparison.Actual == 0) &
    (comparison.Prediction == 0)
].sample(1, random_state=42)
tn = tn[X_test_lr.columns]

fn = comparison[
    (comparison.Actual == 1) &
    (comparison.Prediction == 0)
].sample(1, random_state=42)
fn = fn[X_test_lr.columns]

##### True Positive

In [16]:
coef = pd.Series(
    lr.coef_[0],
    index=X_test_lr.columns
)

contrib = tp.iloc[0] * coef

lr_tp = (
    pd.DataFrame({
        "Feature": contrib.index,
        "Contribution": contrib.values,
        "Importance": np.abs(contrib.values)
    })
    .sort_values("Importance", ascending=False)
    .head(5)
    .drop(columns="Importance")
    .reset_index(drop=True)
)

lr_tp

,Feature,Contribution
0,ext_source_3,1.377675
1,ext_source_2,0.927147
2,ext_source_1,0.557875
3,occupation_type_Accountants,-0.190843
4,ext_source_1_missing,-0.144148


##### True Negative 

In [17]:
coef = pd.Series(
    lr.coef_[0],
    index=X_test_lr.columns
)

contrib = tn.iloc[0] * coef

lr_tn = (
    pd.DataFrame({
        "Feature": contrib.index,
        "Contribution": contrib.values,
        "Importance": np.abs(contrib.values)
    })
    .sort_values("Importance", ascending=False)
    .head(5)
    .drop(columns="Importance")
    .reset_index(drop=True)
)

lr_tn

,Feature,Contribution
0,ext_source_3,-0.666497
1,ext_source_2,0.360374
2,ext_source_1_missing,-0.144148
3,flag_document_3,0.095167
4,days_employed,0.093941


##### False Positive

In [18]:
coef = pd.Series(
    lr.coef_[0],
    index=X_test_lr.columns
)

contrib = fp.iloc[0] * coef

lr_fp = (
    pd.DataFrame({
        "Feature": contrib.index,
        "Contribution": contrib.values,
        "Importance": np.abs(contrib.values)
    })
    .sort_values("Importance", ascending=False)
    .head(5)
    .drop(columns="Importance")
    .reset_index(drop=True)
)

lr_fp

,Feature,Contribution
0,ext_source_2,0.964872
1,ext_source_3,0.467431
2,ext_source_1,0.433564
3,reg_city_not_live_city,0.160622
4,region_rating_client_w_city,0.155384


##### False Negative

In [19]:
coef = pd.Series(
    lr.coef_[0],
    index=X_test_lr.columns
)

contrib = fn.iloc[0] * coef

lr_fn = (
    pd.DataFrame({
        "Feature": contrib.index,
        "Contribution": contrib.values,
        "Importance": np.abs(contrib.values)
    })
    .sort_values("Importance", ascending=False)
    .head(5)
    .drop(columns="Importance")
    .reset_index(drop=True)
)

lr_fn

,Feature,Contribution
0,ext_source_3,0.996064
1,ext_source_2,0.509028
2,organization_type_Self-employed,0.122424
3,name_housing_type_Municipal apartment,0.121177
4,ext_source_1_missing,0.111560


#### Decision Tree

Since the built-in interpretation mechanism of Decision Tree is a decision path, it will be shown in a separate table.

In [20]:
from sklearn.tree import export_text

print(export_text(dec_tree, feature_names=list(X_test_others.columns)))

|--- ext_source_3 <= 0.54
|   |--- ext_source_2 <= 0.41
|   |   |--- ext_source_3 <= 0.24
|   |   |   |--- class: 1
|   |   |--- ext_source_3 >  0.24
|   |   |   |--- ext_source_2 <= 0.15
|   |   |   |   |--- class: 1
|   |   |   |--- ext_source_2 >  0.15
|   |   |   |   |--- ext_source_1 <= 0.54
|   |   |   |   |   |--- class: 1
|   |   |   |   |--- ext_source_1 >  0.54
|   |   |   |   |   |--- class: 0
|   |--- ext_source_2 >  0.41
|   |   |--- ext_source_3 <= 0.26
|   |   |   |--- ext_source_1 <= 0.67
|   |   |   |   |--- class: 1
|   |   |   |--- ext_source_1 >  0.67
|   |   |   |   |--- class: 0
|   |   |--- ext_source_3 >  0.26
|   |   |   |--- ext_source_2 <= 0.62
|   |   |   |   |--- ext_source_1 <= 0.51
|   |   |   |   |   |--- days_employed <= -1463.50
|   |   |   |   |   |   |--- class: 0
|   |   |   |   |   |--- days_employed >  -1463.50
|   |   |   |   |   |   |--- class: 1
|   |   |   |   |--- ext_source_1 >  0.51
|   |   |   |   |   |--- class: 0
|   |   |   |--- ext_sou

In [21]:
# Create predictions
y_pred = dec_tree.predict(X_test_others)

comparison = X_test_others.copy()
comparison["Actual"] = y_test_others.values
comparison["Prediction"] = y_pred

In [22]:
# Randomly sample tp, tn, fp, fn 
tp = comparison[
    (comparison.Actual == 1) &
    (comparison.Prediction == 1)
].sample(1, random_state=42)
tp = tp[X_test_others.columns] # Choose this to remove the "Actual" and "Prediction" columns from the tp dataframe

fp = comparison[
    (comparison.Actual == 0) &
    (comparison.Prediction == 1)
].sample(1, random_state=42)
fp = fp[X_test_others.columns]

tn = comparison[
    (comparison.Actual == 0) &
    (comparison.Prediction == 0)
].sample(1, random_state=42)
tn = tn[X_test_others.columns]

fn = comparison[
    (comparison.Actual == 1) &
    (comparison.Prediction == 0)
].sample(1, random_state=42)
fn = fn[X_test_others.columns]

##### True Positive

In [ ]:
node_indicator = dec_tree.decision_path(tp)
leaf_id = dec_tree.apply(tp)

feature = dec_tree.tree_.feature
threshold = dec_tree.tree_.threshold

node_index = node_indicator.indices[
    node_indicator.indptr[0]:node_indicator.indptr[1]
]

rows = []

for node_id in node_index:

    if leaf_id[0] == node_id:
        continue

    feature_name = X_test_others.columns[feature[node_id]]
    feature_value = tp.iloc[0, feature[node_id]]
    thresh = threshold[node_id]

    direction = "<=" if feature_value <= thresh else ">"

    rows.append({
        "Feature": feature_name,
        "Value": feature_value,
        "Threshold": thresh,
        "Decision": direction
    })

dt_tp = pd.DataFrame(rows)

dt_tp.to_csv('model_results/understandability/dt_tp.csv')

dt_tp

##### True Negative

In [56]:
node_indicator = dec_tree.decision_path(tn)
leaf_id = dec_tree.apply(tn)

feature = dec_tree.tree_.feature
threshold = dec_tree.tree_.threshold

node_index = node_indicator.indices[
    node_indicator.indptr[0]:node_indicator.indptr[1]
]

rows = []

for node_id in node_index:

    if leaf_id[0] == node_id:
        continue

    feature_name = X_test_others.columns[feature[node_id]]
    feature_value = tn.iloc[0, feature[node_id]]
    thresh = threshold[node_id]

    direction = "<=" if feature_value <= thresh else ">"

    rows.append({
        "Feature": feature_name,
        "Value": feature_value,
        "Threshold": thresh,
        "Decision": direction
    })

dt_tn = pd.DataFrame(rows)

dt_tn.to_csv('model_results/understandability/dt_tn.csv')

dt_tn

,Feature,Value,Threshold,Decision
0,ext_source_3,0.780144,0.536173,>
1,ext_source_2,0.487218,0.487890,<=
2,ext_source_2,0.487218,0.253790,>


##### False Positive

In [57]:
node_indicator = dec_tree.decision_path(fp)
leaf_id = dec_tree.apply(fp)

feature = dec_tree.tree_.feature
threshold = dec_tree.tree_.threshold

node_index = node_indicator.indices[
    node_indicator.indptr[0]:node_indicator.indptr[1]
]

rows = []

for node_id in node_index:

    if leaf_id[0] == node_id:
        continue

    feature_name = X_test_others.columns[feature[node_id]]
    feature_value = fp.iloc[0, feature[node_id]]
    thresh = threshold[node_id]

    direction = "<=" if feature_value <= thresh else ">"

    rows.append({
        "Feature": feature_name,
        "Value": feature_value,
        "Threshold": thresh,
        "Decision": direction
    })

dt_fp = pd.DataFrame(rows)

dt_fp.to_csv('model_results/understandability/dt_fp.csv')

dt_fp

,Feature,Value,Threshold,Decision
0,ext_source_3,0.173527,0.536173,<=
1,ext_source_2,0.012695,0.414495,<=
2,ext_source_3,0.173527,0.242524,<=


##### False Negative

In [58]:
node_indicator = dec_tree.decision_path(fn)
leaf_id = dec_tree.apply(fn)

feature = dec_tree.tree_.feature
threshold = dec_tree.tree_.threshold

node_index = node_indicator.indices[
    node_indicator.indptr[0]:node_indicator.indptr[1]
]

rows = []

for node_id in node_index:

    if leaf_id[0] == node_id:
        continue

    feature_name = X_test_others.columns[feature[node_id]]
    feature_value = fn.iloc[0, feature[node_id]]
    thresh = threshold[node_id]

    direction = "<=" if feature_value <= thresh else ">"

    rows.append({
        "Feature": feature_name,
        "Value": feature_value,
        "Threshold": thresh,
        "Decision": direction
    })

dt_fn = pd.DataFrame(rows)

dt_fn.to_csv('model_results/understandability/dt_fn.csv')

dt_fn

,Feature,Value,Threshold,Decision
0,ext_source_3,0.535276,0.536173,<=
1,ext_source_2,0.530230,0.414495,>
2,ext_source_3,0.535276,0.262948,>
3,ext_source_2,0.530230,0.615393,<=
4,ext_source_1,0.506156,0.507184,<=
5,days_employed,-1658.000000,-1463.500000,<=


#### Random Forest

In [27]:
# Create predictions
y_pred = rf.predict(X_test_others)

comparison = X_test_others.copy()
comparison["Actual"] = y_test_others.values
comparison["Prediction"] = y_pred

# Randomly sample tp, tn, fp, fn for SHAP analysis
tp = comparison[
    (comparison.Actual == 1) &
    (comparison.Prediction == 1)
].sample(1, random_state=42)
tp = tp[X_test_others.columns] # Choose this to remove the "Actual" and "Prediction" columns from the tp dataframe

fp = comparison[
    (comparison.Actual == 0) &
    (comparison.Prediction == 1)
].sample(1, random_state=42)
fp = fp[X_test_others.columns]

tn = comparison[
    (comparison.Actual == 0) &
    (comparison.Prediction == 0)
].sample(1, random_state=42)
tn = tn[X_test_others.columns]

fn = comparison[
    (comparison.Actual == 1) &
    (comparison.Prediction == 0)
].sample(1, random_state=42)
fn = fn[X_test_others.columns]

##### Fit SHAP

In [28]:
explainer = shap.TreeExplainer(rf)

##### True Positive

In [29]:
shap_values = explainer(tp)

rf_contrib = pd.Series(
    shap_values.values[0, :, 1],   # class 1
    index=X_test_others.columns
)

rf_tp = (
    pd.DataFrame({
        "Feature": rf_contrib.index,
        "Contribution": rf_contrib.values,
        "Importance": np.abs(rf_contrib.values)
    })
    .sort_values("Importance", ascending=False)
    .head(5)
    .drop(columns="Importance")
    .reset_index(drop=True)
)

rf_tp

,Feature,Contribution
0,ext_source_3,0.073156
1,days_employed,-0.071073
2,days_birth,-0.019542
3,code_gender_M,0.012624
4,code_gender_F,0.011729


##### True Negative

In [30]:
shap_values_tn = explainer(tn)

rf_contrib = pd.Series(
    shap_values_tn.values[0, :, 1],   # class 1
    index=X_test_others.columns
)

rf_tn = (
    pd.DataFrame({
        "Feature": rf_contrib.index,
        "Contribution": rf_contrib.values,
        "Importance": np.abs(rf_contrib.values)
    })
    .sort_values("Importance", ascending=False)
    .head(5)
    .drop(columns="Importance")
    .reset_index(drop=True)
)

rf_tn

,Feature,Contribution
0,amt_annuity,-0.037021
1,ext_source_2,-0.032597
2,days_id_publish,0.019243
3,flag_document_3,-0.012189
4,elevators_avg,-0.011901


##### False Positive

In [31]:
shap_values_fp = explainer(fp)

rf_contrib = pd.Series(
    shap_values_fp.values[0, :, 1],   # class 1
    index=X_test_others.columns
)

rf_fp = (
    pd.DataFrame({
        "Feature": rf_contrib.index,
        "Contribution": rf_contrib.values,
        "Importance": np.abs(rf_contrib.values)
    })
    .sort_values("Importance", ascending=False)
    .head(5)
    .drop(columns="Importance")
    .reset_index(drop=True)
)

rf_fp

,Feature,Contribution
0,ext_source_3,0.073905
1,ext_source_2,0.048051
2,name_education_type_Higher education,-0.036774
3,name_education_type_Secondary / secondary special,-0.024649
4,days_employed,0.013928


##### False Negative

In [32]:
shap_values_fn = explainer(fn)

rf_contrib = pd.Series(
    shap_values_fn.values[0, :, 1],   # class 1
    index=X_test_others.columns
)

rf_fn = (
    pd.DataFrame({
        "Feature": rf_contrib.index,
        "Contribution": rf_contrib.values,
        "Importance": np.abs(rf_contrib.values)
    })
    .sort_values("Importance", ascending=False)
    .head(5)
    .drop(columns="Importance")
    .reset_index(drop=True)
)

rf_fn

,Feature,Contribution
0,ext_source_2,0.061258
1,ext_source_3,0.059112
2,amt_credit,-0.042442
3,name_education_type_Higher education,-0.035418
4,name_education_type_Secondary / secondary special,-0.028205


#### XGBoost

In [33]:
# Create predictions
y_pred = xgb.predict(X_test_others)

comparison = X_test_others.copy()
comparison["Actual"] = y_test_others.values
comparison["Prediction"] = y_pred

# Randomly sample tp, tn, fp, fn for SHAP analysis
tp = comparison[
    (comparison.Actual == 1) &
    (comparison.Prediction == 1)
].sample(1, random_state=42)
tp = tp[X_test_others.columns] # Choose this to remove the "Actual" and "Prediction" columns from the tp dataframe

fp = comparison[
    (comparison.Actual == 0) &
    (comparison.Prediction == 1)
].sample(1, random_state=42)
fp = fp[X_test_others.columns]

tn = comparison[
    (comparison.Actual == 0) &
    (comparison.Prediction == 0)
].sample(1, random_state=42)
tn = tn[X_test_others.columns]

fn = comparison[
    (comparison.Actual == 1) &
    (comparison.Prediction == 0)
].sample(1, random_state=42)
fn = fn[X_test_others.columns]

##### Fit SHAP

In [34]:
explainer_xgb = shap.TreeExplainer(xgb)

##### True Positive

In [35]:
shap_values = explainer_xgb(tp)

xgb_contrib = pd.Series(
    shap_values.values[0],   # class 1
    index=X_test_others.columns
)

xgb_tp = (
    pd.DataFrame({
        "Feature": xgb_contrib.index,
        "Contribution": xgb_contrib.values,
        "Importance": np.abs(xgb_contrib.values)
    })
    .sort_values("Importance", ascending=False)
    .head(5)
    .drop(columns="Importance")
    .reset_index(drop=True)
)

xgb_tp

,Feature,Contribution
0,ext_source_3,0.579517
1,ext_source_1,-0.394450
2,days_birth,-0.269580
3,def_60_cnt_social_circle,0.200926
4,reg_city_not_live_city,0.156012


##### True Negative

In [36]:
shap_values = explainer_xgb(tn)

xgb_contrib = pd.Series(
    shap_values.values[0],   # class 1
    index=X_test_others.columns
)

xgb_tn = (
    pd.DataFrame({
        "Feature": xgb_contrib.index,
        "Contribution": xgb_contrib.values,
        "Importance": np.abs(xgb_contrib.values)
    })
    .sort_values("Importance", ascending=False)
    .head(5)
    .drop(columns="Importance")
    .reset_index(drop=True)
)

xgb_tn

,Feature,Contribution
0,amt_credit,-0.371687
1,ext_source_2,-0.369320
2,days_employed,0.172349
3,amt_annuity,-0.131557
4,flag_own_car,-0.109675


##### False Positive

In [37]:
shap_values = explainer_xgb(fp)

xgb_contrib = pd.Series(
    shap_values.values[0],   # class 1
    index=X_test_others.columns
)

xgb_fp = (
    pd.DataFrame({
        "Feature": xgb_contrib.index,
        "Contribution": xgb_contrib.values,
        "Importance": np.abs(xgb_contrib.values)
    })
    .sort_values("Importance", ascending=False)
    .head(5)
    .drop(columns="Importance")
    .reset_index(drop=True)
)

xgb_fp

,Feature,Contribution
0,ext_source_3,0.942052
1,ext_source_2,0.449077
2,def_60_cnt_social_circle,0.218827
3,days_birth,-0.218589
4,name_contract_type_Cash loans,-0.166384


##### False Negative

In [38]:
shap_values = explainer_xgb(fn)

xgb_contrib = pd.Series(
    shap_values.values[0],   # class 1
    index=X_test_others.columns
)

xgb_fn = (
    pd.DataFrame({
        "Feature": xgb_contrib.index,
        "Contribution": xgb_contrib.values,
        "Importance": np.abs(xgb_contrib.values)
    })
    .sort_values("Importance", ascending=False)
    .head(5)
    .drop(columns="Importance")
    .reset_index(drop=True)
)

xgb_fn

,Feature,Contribution
0,days_id_publish,0.358429
1,amt_credit,-0.324502
2,days_employed,-0.256199
3,ext_source_2,-0.196956
4,days_birth,0.126041


#### LightGBM

In [39]:
# Create predictions
y_pred = lgbm_sm.predict(X_test_others)

comparison = X_test_others.copy()
comparison["Actual"] = y_test_others.values
comparison["Prediction"] = y_pred

# Randomly sample tp, tn, fp, fn for SHAP analysis
tp = comparison[
    (comparison.Actual == 1) &
    (comparison.Prediction == 1)
].sample(1, random_state=42)
tp = tp[X_test_others.columns] # Choose this to remove the "Actual" and "Prediction" columns from the tp dataframe

fp = comparison[
    (comparison.Actual == 0) &
    (comparison.Prediction == 1)
].sample(1, random_state=42)
fp = fp[X_test_others.columns]

tn = comparison[
    (comparison.Actual == 0) &
    (comparison.Prediction == 0)
].sample(1, random_state=42)
tn = tn[X_test_others.columns]

fn = comparison[
    (comparison.Actual == 1) &
    (comparison.Prediction == 0)
].sample(1, random_state=42)
fn = fn[X_test_others.columns]

##### Fit SHAP

In [40]:
explainer_lgbm = shap.TreeExplainer(lgbm_sm)

##### True Positive

In [41]:
shap_values = explainer_lgbm(tp)

lgbm_contrib = pd.Series(
    shap_values.values[0],   # class 1
    index=X_test_others.columns
)

lgbm_tp = (
    pd.DataFrame({
        "Feature": lgbm_contrib.index,
        "Contribution": lgbm_contrib.values,
        "Importance": np.abs(lgbm_contrib.values)
    })
    .sort_values("Importance", ascending=False)
    .head(5)
    .drop(columns="Importance")
    .reset_index(drop=True)
)

lgbm_tp

,Feature,Contribution
0,ext_source_3,0.931741
1,ext_source_2,0.561474
2,ext_source_1_missing,-0.457108
3,ext_source_1,0.266910
4,ext_source_3_missing,-0.249958


##### True Negative

In [42]:
shap_values = explainer_lgbm(tn)

lgbm_contrib = pd.Series(
    shap_values.values[0],   # class 1
    index=X_test_others.columns
)

lgbm_tn = (
    pd.DataFrame({
        "Feature": lgbm_contrib.index,
        "Contribution": lgbm_contrib.values,
        "Importance": np.abs(lgbm_contrib.values)
    })
    .sort_values("Importance", ascending=False)
    .head(5)
    .drop(columns="Importance")
    .reset_index(drop=True)
)

lgbm_tn

,Feature,Contribution
0,ext_source_3,-0.795295
1,name_income_type_Working,-0.515020
2,flag_document_3,-0.340439
3,ext_source_3_missing,-0.288904
4,code_gender_F,-0.251734


##### False Positive

In [43]:
shap_values = explainer_lgbm(fp)

lgbm_contrib = pd.Series(
    shap_values.values[0],   # class 1
    index=X_test_others.columns
)

lgbm_fp = (
    pd.DataFrame({
        "Feature": lgbm_contrib.index,
        "Contribution": lgbm_contrib.values,
        "Importance": np.abs(lgbm_contrib.values)
    })
    .sort_values("Importance", ascending=False)
    .head(5)
    .drop(columns="Importance")
    .reset_index(drop=True)
)

lgbm_fp

,Feature,Contribution
0,ext_source_2,0.788417
1,ext_source_3,0.753594
2,ext_source_3_missing,-0.266804
3,name_family_status_Married,-0.208218
4,code_gender_F,-0.183215


##### False Negative

In [44]:
shap_values = explainer_lgbm(fn)

lgbm_contrib = pd.Series(
    shap_values.values[0],   # class 1
    index=X_test_others.columns
)

lgbm_fn = (
    pd.DataFrame({
        "Feature": lgbm_contrib.index,
        "Contribution": lgbm_contrib.values,
        "Importance": np.abs(lgbm_contrib.values)
    })
    .sort_values("Importance", ascending=False)
    .head(5)
    .drop(columns="Importance")
    .reset_index(drop=True)
)

lgbm_fn

,Feature,Contribution
0,ext_source_3,-0.915899
1,code_gender_M,-0.347814
2,ext_source_3_missing,0.274146
3,name_income_type_Working,-0.206150
4,name_family_status_Married,-0.195599


#### Merge all mdodels

In [ ]:
def create_comparison_table(results, decimals=3):
    """
    results = {
        "LR": lr_tp,
        "RF": rf_tp,
        "XGB": xgb_tp,
        "LGBM": lgbm_tp
    }

    Each dataframe must contain:
        Feature
        Contribution

    already sorted descending by importance.
    """

    n = len(next(iter(results.values())))

    table = pd.DataFrame({
        "Rank": range(1, n + 1)
    })

    for model, df in results.items():

        table[f"{model} Feature"] = df["Feature"].values

        table[f"{model} Contribution"] = (
            df["Contribution"]
            .round(decimals)
            .values
        )

    return table

In [ ]:
tp_table = create_comparison_table({

    "LR": lr_tp,
    "RF": rf_tp,
    "XGB": xgb_tp,
    "LGBM": lgbm_tp

})

ValueError: Length of values (3) does not match length of index (5)

In [ ]:
tn_table = create_comparison_table({

    "LR": lr_tn,
    "RF": rf_tn,
    "XGB": xgb_tn,
    "LGBM": lgbm_tn

})

In [ ]:
fp_table = create_comparison_table({

    "LR": lr_fp,
    "RF": rf_fp,
    "XGB": xgb_fp,
    "LGBM": lgbm_fp

})

In [ ]:
fn_table = create_comparison_table({

    "LR": lr_fn,
    "RF": rf_fn,
    "XGB": xgb_fn,
    "LGBM": lgbm_fn

})

#### Export all tables in Excel

In [ ]:
with pd.ExcelWriter(
    "Understandability_Tables.xlsx",
    engine="openpyxl"
) as writer:

    tp_table.to_excel(writer,
                      sheet_name="True Positive",
                      index=False)

    tn_table.to_excel(writer,
                      sheet_name="True Negative",
                      index=False)

    fp_table.to_excel(writer,
                      sheet_name="False Positive",
                      index=False)

    fn_table.to_excel(writer,
                      sheet_name="False Negative",
                      index=False)